# 02 — Data Collection & Manager Attribution

**CRISP-DM phase:** Data Preparation.

Builds the manager-attributed player x match dataset for the three managers, following the verified tenure windows in `data/external/manager_tenures.md` and the scope decided in `outputs/PHASE0_DELIVERABLE.md` (Manchester City / Liverpool / Sassuolo+Benevento+Brighton+Marseille, 2017/18-2025/26 as available).

**Hard rule enforced here (brief SS12-15):** every row is attributed to a manager using match-date windows, not season labels, so mid-season manager changes (Benevento, Brighton, Marseille) are handled correctly by construction rather than by post-hoc filtering.

Outputs:
- `data/processed/player_match_manager.parquet` — player x match, manager-attributed
- `data/processed/player_manager_season.parquet` — aggregated player x manager x season (brief SS16)
- `outputs/audit/integrity_audit.md` — the SS49-style audit table

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from data_sources import download_fbref, FBREF_STAT_TYPES, DATA_PROCESSED
from manager_attribution import (
    MANAGER_WINDOWS_DF,
    build_player_match_dataset,
    integrity_audit,
    load_tenure_table,
    save_processed,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## 1. Confirm source data is present

Downloads are cached under `data/raw/fbref_match/` (see `outputs/audit/data_availability_matrix.md` for the source audit). Re-running this cell is a no-op if files already exist.

In [2]:
manifest = download_fbref(leagues=["ENG", "ITA", "FRA"], stat_types=FBREF_STAT_TYPES)
manifest

,league,stat_type,status,mb
0,ENG,summary,cached,4.79
1,ENG,passing,cached,48.13
2,ENG,passing_types,cached,45.79
3,ENG,possession,cached,46.30
4,ENG,defense,cached,45.69
5,ENG,misc,cached,44.55
6,ENG,keeper,cached,3.30
7,ITA,summary,cached,4.04
8,ITA,passing,cached,43.96
9,ITA,passing_types,cached,42.95


## 2. Manager attribution windows

The window boundaries encoded here are the tighter of (a) the verified tenure dates in `manager_tenures.csv` and (b) the FBref match-level data-availability window (2017/18 season onward). See `src/manager_attribution.py::MANAGER_WINDOWS` and `decision_log.md` D-004/D-005 for the reasoning behind each boundary, especially the mid-season ones (Benevento, Brighton, Marseille).

In [3]:
MANAGER_WINDOWS_DF

,manager,team,window_start,window_end,tenure_row,confidence
0,Pep Guardiola,Manchester City,2017-07-01,2024-07-01,G4,high
1,Jürgen Klopp,Liverpool,2017-07-01,2024-07-01,K3,high
2,Roberto De Zerbi,Sassuolo,2018-07-01,2021-07-01,D5,high
3,Roberto De Zerbi,Benevento,2017-10-23,2018-07-01,D4,low
4,Roberto De Zerbi,Brighton & Hove Albion,2022-10-01,2024-07-01,D7,high
5,Roberto De Zerbi,Marseille,2024-07-01,2026-02-09,D8,high


## 3. Build the manager-attributed player x match dataset

Merges all 7 FBref stat blocks (summary, passing, passing_types, possession, defense, misc, keeper) on `(MatchURL, Player, Team, Season_End_Year)`, restricted to each manager's verified window. Stat-block-specific columns are prefixed (e.g. `passing__PrgP`) to avoid silent collisions between blocks that republish the same metric name (e.g. `Int` appears in both `defense` and `misc`).

In [4]:
player_match = build_player_match_dataset()
print(player_match.shape)
player_match.head()

(10996, 267)


,MatchURL,League,Match_Date,Matchweek,Home_Team,Home_Formation,Home_Score,Home_xG,Home_Goals,Home_Yellow_Cards,Home_Red_Cards,Away_Team,Away_Formation,Away_Score,Away_xG,Away_Goals,Away_Yellow_Cards,Away_Red_Cards,Game_URL,Team,...,summary__Att_Passes,summary__Cmp_percent_Passes,summary__PrgP_Passes,summary__Carries_Carries,summary__PrgC_Carries,summary__Att_Take_Ons,summary__Succ_Take_Ons,summary__Fls,summary__Fld,summary__Off,summary__Crs,summary__TklW,summary__OG,summary__PKwon,summary__PKcon,possession__Def 3rd_Tackles,possession__Mid 3rd_Tackles,possession__Att 3rd_Tackles,possession__Tkl+Int,misc__X2CrdY
0,https://fbref.com/en/matches/00a73645/Manchest...,Premier League,2023-04-26,Premier League (Matchweek 33),Manchester City,4-3-3,4,2.5,Kevin De Bruyne · 7&rsquor; John Stones · 45+1...,3,0,Arsenal,4-3-3,1,0.5,Rob Holding · 86&rsquor;,1,0,https://fbref.com/en/matches/00a73645/Manchest...,Manchester City,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://fbref.com/en/matches/00a73645/Manchest...,Premier League,2023-04-26,Premier League (Matchweek 33),Manchester City,4-3-3,4,2.5,Kevin De Bruyne · 7&rsquor; John Stones · 45+1...,3,0,Arsenal,4-3-3,1,0.5,Rob Holding · 86&rsquor;,1,0,https://fbref.com/en/matches/00a73645/Manchest...,Manchester City,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://fbref.com/en/matches/00a73645/Manchest...,Premier League,2023-04-26,Premier League (Matchweek 33),Manchester City,4-3-3,4,2.5,Kevin De Bruyne · 7&rsquor; John Stones · 45+1...,3,0,Arsenal,4-3-3,1,0.5,Rob Holding · 86&rsquor;,1,0,https://fbref.com/en/matches/00a73645/Manchest...,Manchester City,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://fbref.com/en/matches/00a73645/Manchest...,Premier League,2023-04-26,Premier League (Matchweek 33),Manchester City,4-3-3,4,2.5,Kevin De Bruyne · 7&rsquor; John Stones · 45+1...,3,0,Arsenal,4-3-3,1,0.5,Rob Holding · 86&rsquor;,1,0,https://fbref.com/en/matches/00a73645/Manchest...,Manchester City,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://fbref.com/en/matches/00a73645/Manchest...,Premier League,2023-04-26,Premier League (Matchweek 33),Manchester City,4-3-3,4,2.5,Kevin De Bruyne · 7&rsquor; John Stones · 45+1...,3,0,Arsenal,4-3-3,1,0.5,Rob Holding · 86&rsquor;,1,0,https://fbref.com/en/matches/00a73645/Manchest...,Manchester City,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Integrity audit (brief SS49)

Must be reviewed and pass before any modelling proceeds. Checks: matches/players/minutes per manager-team-season, duplicate player-match rows, and rows whose `Match_Date` falls outside the stamped attribution window (should always be zero, since the filtering happens at construction time — this is a construction-correctness check, not a discovery step).

In [5]:
audit = integrity_audit(player_match)
audit

,manager,team,Season_End_Year,tenure_row,attribution_confidence,matches,players,player_match_rows,total_minutes,earliest_match,latest_match,duplicate_rows_flag,rows_outside_window
0,Jürgen Klopp,Liverpool,2018,K3,high,38,27,529,37566.0,2017-08-12,2018-05-13,False,0
1,Jürgen Klopp,Liverpool,2019,K3,high,38,23,527,37609.0,2018-08-12,2019-05-12,False,0
2,Jürgen Klopp,Liverpool,2020,K3,high,38,24,545,37605.0,2019-08-09,2020-07-26,False,0
3,Jürgen Klopp,Liverpool,2021,K3,high,38,28,522,37620.0,2020-09-12,2021-05-23,False,0
4,Jürgen Klopp,Liverpool,2022,K3,high,38,27,529,37606.0,2021-08-14,2022-05-22,False,0
5,Jürgen Klopp,Liverpool,2023,K3,high,38,28,582,37586.0,2022-08-06,2023-05-28,False,0
6,Jürgen Klopp,Liverpool,2024,K3,high,38,31,584,37434.0,2023-08-13,2024-05-19,False,0
7,Pep Guardiola,Manchester City,2018,G4,high,38,25,524,37571.0,2017-08-12,2018-05-13,False,0
8,Pep Guardiola,Manchester City,2019,G4,high,38,21,526,37618.0,2018-08-12,2019-05-12,False,0
9,Pep Guardiola,Manchester City,2020,G4,high,38,24,538,37492.0,2019-08-10,2020-07-26,False,0


In [6]:
assert audit.rows_outside_window.sum() == 0, "Attribution-window construction bug: rows fall outside their stamped manager window"
assert not audit.duplicate_rows_flag.any(), "Duplicate player-match rows detected within a manager-team-season group"
print("Integrity audit passed: no out-of-window rows, no duplicate player-match rows.")

Integrity audit passed: no out-of-window rows, no duplicate player-match rows.


## 5. Cross-check against StatsBomb's own manager field (independent source)

StatsBomb Open Data records the manager in charge of each team directly in its match metadata (see `data_sources.py::sb_match_manager_index`). Where its coverage overlaps our in-scope window, this is an independent, second-source confirmation of the tenure table — not a substitute for it, since StatsBomb's own open-data coverage of these exact team-seasons is otherwise too sparse to use as a primary source (see `outputs/audit/data_availability_matrix.md`).

In [7]:
from data_sources import sb_match_manager_index

sb_idx = sb_match_manager_index()
overlap = sb_idx[sb_idx.team.isin(["Manchester City", "Liverpool", "Brighton & Hove Albion", "Sassuolo", "Benevento"])]
overlap_in_scope = overlap[overlap.match_date >= "2017-07-01"]
print(f"StatsBomb rows for in-scope teams, 2017-07-01 onward: {len(overlap_in_scope)}")
overlap_in_scope[["match_date", "team", "manager", "competition", "season"]].sort_values("match_date") if len(overlap_in_scope) else "No overlap available in StatsBomb's curated coverage for this window — expected, per the data availability audit (StatsBomb has sparse coverage outside Guardiola's Barcelona era). Not a failure of this check."

StatsBomb rows for in-scope teams, 2017-07-01 onward: 2


,match_date,team,manager,competition,season
3371,2018-05-26,Liverpool,Jürgen Klopp,Champions League,2017/2018
3397,2019-06-01,Liverpool,Jürgen Klopp,Champions League,2018/2019


## 6. Aggregated player x manager x season table (brief SS16)

Maintained alongside the match-level table so both aggregation strategies can be investigated downstream, per the brief's explicit recommendation.

In [8]:
numeric_cols = player_match.select_dtypes("number").columns.tolist()
group_cols = ["Player", "Player_Href", "manager", "team", "Season_End_Year", "tenure_row", "attribution_confidence", "Pos"]
agg_dict = {c: "sum" for c in numeric_cols if c not in ("Season_End_Year",)}
player_manager_season = player_match.groupby(group_cols, dropna=False).agg(agg_dict).reset_index()
player_manager_season["matches_played"] = player_match.groupby(group_cols, dropna=False).size().values
print(player_manager_season.shape)
player_manager_season.head()

(2711, 246)


/var/folders/jz/y19hl2rx6gz8fhcs7qnq2pvw0000gn/T/ipykernel_10116/2647581786.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  player_manager_season = player_match.groupby(group_cols, dropna=False).agg(agg_dict).reset_index()
/var/folders/jz/y19hl2rx6gz8fhcs7qnq2pvw0000gn/T/ipykernel_10116/2647581786.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  player_manager_season = player_match.groupby(group_cols, dropna=False).agg(agg_dict).reset_index()
/var/folders/jz/y19hl2rx6gz8fhcs7qnq2pvw0000gn/T/ipykernel_10116/2647581786.py:4

,Player,Player_Href,manager,team,Season_End_Year,tenure_row,attribution_confidence,Pos,Home_Score,Home_xG,Home_Yellow_Cards,Home_Red_Cards,Away_Score,Away_xG,Away_Yellow_Cards,Away_Red_Cards,Player_Num,Min,passing__Cmp_Total,passing__Att_Total,...,summary__Cmp_percent_Passes,summary__PrgP_Passes,summary__Carries_Carries,summary__PrgC_Carries,summary__Att_Take_Ons,summary__Succ_Take_Ons,summary__Fls,summary__Fld,summary__Off,summary__Crs,summary__TklW,summary__OG,summary__PKwon,summary__PKcon,possession__Def 3rd_Tackles,possession__Mid 3rd_Tackles,possession__Att 3rd_Tackles,possession__Tkl+Int,misc__X2CrdY,matches_played
0,Achraf Lazaar,/en/players/957d41d2/Achraf-Lazaar,Roberto De Zerbi,Benevento,2018,D4,low,LM,2,2.0,5,0,1,1.3,1,0,77.0,90.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,Achraf Lazaar,/en/players/957d41d2/Achraf-Lazaar,Roberto De Zerbi,Benevento,2018,D4,low,RM,1,1.4,5,1,2,2.9,1,0,77.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,Achraf Lazaar,/en/players/957d41d2/Achraf-Lazaar,Roberto De Zerbi,Benevento,2018,D4,low,WB,3,3.7,4,0,6,2.9,4,0,154.0,151.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
3,Adam Lallana,/en/players/99813635/Adam-Lallana,Jürgen Klopp,Liverpool,2018,K3,high,AM,1,1.3,0,0,2,1.7,0,0,20.0,85.0,35.0,46.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,Adam Lallana,/en/players/99813635/Adam-Lallana,Jürgen Klopp,Liverpool,2018,K3,high,"CM,AM",1,0.2,0,0,0,2.5,2,0,20.0,23.0,26.0,31.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


## 7. Save processed outputs

In [9]:
path1 = save_processed(player_match, "player_match_manager")
path2 = save_processed(player_manager_season, "player_manager_season")
print(path1)
print(path2)

audit_md_path = Path.cwd().parent / "outputs" / "audit" / "integrity_audit.md"
audit_md_path.parent.mkdir(parents=True, exist_ok=True)
with open(audit_md_path, "w") as f:
    f.write("# Data Integrity Audit — Manager-Attributed Player x Match Dataset\n\n")
    f.write(f"Generated from `notebooks/02_data_collection_and_manager_attribution.ipynb`.\n\n")
    f.write(audit.to_markdown(index=False))
    f.write("\n\n**Checks passed:** no rows outside their stamped attribution window; no duplicate player-match rows within any manager-team-season group.\n")
print(audit_md_path)

data/processed/player_match_manager.parquet
data/processed/player_manager_season.parquet
outputs/audit/integrity_audit.md
